# Custom Serializers

So far we have been happy with the way Pydantic serializes field values.

But sometimes, especially with certain data types, like datetimes, we may want to control how fields get serialized.

A typical example is to specify how a date or datetime object might get serialized.

Another example might be standardizing the number of decimal places used for floats.

Whatever your need is, you can control how field data gets serialized very easily.

We'll need to use a **decorator** function provided by Pydantic, called `@field_serializer` which is used to control serialization at the field level.

In [ ]:
from pydantic import BaseModel, field_serializer

The decorator has several arguments that defines which field the serializer applies to and how the serializer needs to be applied.

One important option is:
- `when_used`: by default the custom serializer is always used, but we have other options available:
    - `always`: the default, serializer is executed when serializing either to a dict or to JSON
    - `unless-none`: serializer is not used if the value is None
    - `json`: serializer is only used when serializing to JSON
    - `json-unless-none`: serializer used when serializing to JSON, unless the value is None

There is also another option for mode plain vs wrap, but this is rarely used, and I won't cover it in this course.

Let's take a look the `when_used` option and understand the circumstances when our serializer gets called.

In [ ]:
from datetime import datetime


class Model(BaseModel):
    dt: datetime | None = None

    @field_serializer("dt", when_used="always")
    def serialize_name(self, value):
        print(f"type = {type(value)}")
        return value

In [ ]:
m = Model(dt="2020-01-01T12:00:00")
m

So, first thing to realize is that the serializer will run once the model has been populated - which means that `value` in our arguments will be of the valid type (or `None` in our specific example since we made the field nullable).

In [ ]:
m.model_dump()

As you can see, our serializer was invoked - the return value is what will be used in the serialized output.

Let's serialize to JSON and see what happens:

In [ ]:
m.model_dump_json()

The data was correctly serialized, since Pydantic can correctly serialize datetime to JSON (uses ISO standard).

However, when serializing to JSON we may not want this datetime representation. 

We'll get back to that in a minute.

Let's see what happens if the value of `dt` is None:

In [ ]:
m = Model()
m

In [ ]:
m.model_dump()

In [ ]:
m.model_dump_json()

As you can see, our custom serializer was called in both cases.

If we don't want to run our custom serializer when the field value is `None`, we can use one of the other `when_used` options:

In [ ]:
from datetime import datetime


class Model(BaseModel):
    dt: datetime | None = None

    @field_serializer("dt", when_used="unless-none")
    def serialize_name(self, value):
        print(f"type = {type(value)}")
        return value

In [ ]:
m = Model(dt="2020-01-01T12:00:00")
m

In [ ]:
m.model_dump()

In [ ]:
m = Model()
m

In [ ]:
m.model_dump()

In [ ]:
m.model_dump_json()

As you can see, our serializer did not get called when `dt` was `None`.

Let's go back to the case where we only want to change the serialization when serializing to JSON. We might be OK with the dictionary serialization, but for our JSON output we want to modify the datetime format to be formatted like this:

```
2020/1/1 12:00 PM
```

We can use the `strftime()` method to do this:

In [ ]:
dt = datetime(2020, 1, 1, 12, 0, 0)
dt.isoformat()

In [ ]:
dt.strftime("%Y/%#m/%#d %I:%M %p")

So, let's use this in our serializer, and configure the serializer to only apply to JSON serialization, and not when the value is None:

In [ ]:
from datetime import datetime


class Model(BaseModel):
    dt: datetime | None = None

    @field_serializer("dt", when_used="json-unless-none")
    def serialize_name(self, value):
        print(f"type = {type(value)}")
        return value.strftime("%Y/%-m/%-d %I:%M %p")

In [ ]:
m = Model(dt="2020-01-01T12:00:00")
m

In [ ]:
m.model_dump()

As you can see, serializing to a dictionary did not run our serializer.

However, when serializing to JSON:

In [ ]:
m.model_dump_json()

And, because of our configuration, the serializer will not be invoked if the value is `None`:

In [ ]:
m = Model()
m

In [ ]:
m.model_dump_json()

Now suppose we want to implement a different serialization depending on whether we are serializing to a dictionary or to JSON.

We need to somehow be able to figure out, inside our serializer which serialization we are performing and react accordingly.

Pydantic implements yet another argument that we can add to our serializer function - an argument with type `FieldSerializationInfo`. Let's take a look:

In [ ]:
from pydantic import FieldSerializationInfo

In [ ]:
class Model(BaseModel):
    dt: datetime | None = None

    @field_serializer("dt", when_used="unless-none")
    def dt_serializer(self, value, info: FieldSerializationInfo):
        print(f"info={info}")
        return value

In [ ]:
m = Model(dt=datetime(2020, 1, 1))
m

In [ ]:
m.model_dump()

Notice that `mode` value in the `info` object? It is set to `python`.

Now, let's dump to JSON:

In [ ]:
m.model_dump_json()

Notice that the `mode` is now set to `json`.

We could use that, but `FieldSerializationInfo` offers us a method named `mode_is_json` that we can use instead.

In [ ]:
class Model(BaseModel):
    dt: datetime | None = None

    @field_serializer("dt", when_used="unless-none")
    def dt_serializer(self, value, info: FieldSerializationInfo):
        print(f"mode_is_json={info.mode_is_json()}")
        return value

In [ ]:
m = Model(dt=datetime(2020, 1, 1))

In [ ]:
m.model_dump()

In [ ]:
m.model_dump_json()

Let's look at a situation where we might want this flexibility.

Let's say we want our serializer to ensure that datetime objects are always serialized to timezone aware UTC times. Furthermore, we want the serialized value to use the `Z` notation for UTC times, instead of `+00:00` that Python's `isoformat()` function usually returns.

We can easily write Python code to do this, using the `pytz`library.

To complete this example, you'll need to make sure you have `pytz` installed in your virtual environment.

Let's write a simple Python function that will do the following, given a datetime object as an argument:
- if the datetime is naive, make it aware, and assume the naive datetime was already UTC
- if the datetime is aware, change it to be UTC

In [ ]:
import pytz


def make_utc(dt: datetime) -> datetime:
    if dt.tzinfo is None:
        dt = pytz.utc.localize(dt)
    else:
        dt = dt.astimezone(pytz.utc)
    return dt

We can use it this way:

In [ ]:
dt = make_utc(datetime.now())
dt

In [ ]:
dt.isoformat()

We need to change the serialized format of this datetime, and since we know it will always be in UTC, this is quite simple:

In [ ]:
dt.strftime("%Y-%m-%dT%H:%M:%SZ")

Let's make a function for this:

In [ ]:
def dt_utc_json_serializer(dt: datetime) -> str:
    dt = make_utc(dt)
    return dt.strftime("%Y-%m-%dT%H:%M:%SZ")

And now let's implement this in our custom serializer:

In [ ]:
class Model(BaseModel):
    dt: datetime | None = None

    @field_serializer("dt", when_used="unless-none")
    def dt_serializer(self, dt, info: FieldSerializationInfo):
        if info.mode_is_json():
            return dt_utc_json_serializer(dt)
        return make_utc(dt)

In [ ]:
m = Model(dt=datetime(2020, 1, 1))
m

In [ ]:
m.model_dump()

In [ ]:
m.model_dump_json()

And if we have an aware datetime that is not in UTC already:

In [ ]:
eastern = pytz.timezone("US/Eastern")
dt = eastern.localize(datetime(2020, 1, 1))
dt

Now let's use it in our model:

In [ ]:
m = Model(dt=dt)
m

In [ ]:
m.model_dump()

In [ ]:
m.model_dump_json()

In [ ]:
from pydantic import BaseModel, field_serializer, FieldSerializationInfo


class Product(BaseModel):
    name: str
    price_in_toman: int

    @field_serializer("price_in_toman")
    def format_currency(self, price: int, info: FieldSerializationInfo):

        context = info.context or {}
        currency = context.get("currency", "toman")

        if currency == "dollar":
            return f"${price / 50000:.2f}"  # مثلا دلار ۵۰ تومنی!

        return f"{price:,} تومان"


# --- استفاده در اپلیکیشن ---
p = Product(name="FastAPI Course", price_in_toman=1000000)

# ۱. خروجی معمولی برای کاربر ایرانی
print(p.model_dump(context={"currency": "toman"}))
# {'name': 'FastAPI Course', 'price_in_toman': '1,000,000 تومان'}

# ۲. خروجی برای کاربر خارجی
print(p.model_dump(context={"currency": "dollar"}))
# {'name': 'FastAPI Course', 'price_in_ِDollar': '$20.00'}

In [ ]:
from pydantic import BaseModel, field_serializer, FieldSerializationInfo


class PaymentResponse(BaseModel):
    amount: int
    card_number: str

    @field_serializer("card_number")
    def serialize_card(self, card_number: str, info: FieldSerializationInfo) -> str:
        # بررسی نقش کاربر از روی context
        context = info.context or {}
        user_role = context.get("role", "public")

        if user_role == "admin":
            return card_number  # ادمین کارت کامل رو میبینه

        # کاربر عادی کارت ماسک شده میبینه
        return f"{card_number[:4]}-****-****-{card_number[-4:]}"


# تست:
payment = PaymentResponse(amount=150000, card_number="6037991812345678")

# خروجی برای کاربر عادی:
print(payment.model_dump(context={"role": "user"}))
# {'amount': 150000, 'card_number': '6037-****-****-5678'}

# خروجی برای ادمین سیستم:
print(payment.model_dump(context={"role": "admin"}))
# {'amount': 150000, 'card_number': '6037991812345678'}

In [ ]:
from pydantic import BaseModel, Field, field_serializer, FieldSerializationInfo


class UserProfile(BaseModel):
    username: str
    avatar_path: str = Field(..., alias="avatar")

    @field_serializer("avatar_path")
    def serialize_avatar(self, avatar_path: str, info: FieldSerializationInfo) -> str:
        # دریافت دامین به صورت پویا از context در زمان dump
        context = info.context or {}
        base_url = context.get("cdn_url", "https://default-cdn.com")
        return f"{base_url}/{avatar_path}"


# تست در محیط واقعی:
user = UserProfile(username="farshid", avatar="profiles/farshid.jpg")

# موقع خروجی گرفتن، کانفگ CDN رو پاس می‌دیم
json_output = user.model_dump_json(context={"cdn_url": "https://cdn.aparat.com"})
print(json_output)
# خروجی: {"username":"farshid","avatar_path":"https://cdn.aparat.com/profiles/farshid.jpg"}